## New LangChain Chroma vector DB (fresh)

This notebook:
- Loads all PDFs from `documents/`
- Splits into `chunks`
- Builds a **new** Chroma vector DB using LangChain (`langchain_chroma.Chroma.from_documents`)
- Runs a quick retrieval test

It uses a new `persist_directory` + `collection_name` so it does not reuse your existing DB.

### Install (first time only)

If you don't have these already, install them once:

In [ ]:
# %pip install -U langchain-chroma langchain-community langchain-text-splitters pypdf sentence-transformers


### Load PDFs and split into chunks

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_all_pdfs(pdf_directory: str):
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} pdf files")

    all_documents = []
    for pdf_file in pdf_files:
        print(f"processing {pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"
        all_documents.extend(documents)
        print(f"loaded {len(documents)} pages from {pdf_file.name}")

    return all_documents


def split_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    split_docs = splitter.split_documents(documents)
    print(f"split {len(split_docs)} chunks from {len(documents)} pages")
    return split_docs


all_pdf_documents = load_all_pdfs("documents/")
chunks = split_documents(all_pdf_documents)
len(chunks)


### Build a NEW Chroma vector DB (LangChain)

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Same model dimension as `sentence-transformers/all-MiniLM-L6-v2`
lc_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# NEW DB location + NEW collection name (so this does not reuse your old DB)
persist_directory = "vector_stores_new/"
collection_name = "pdf_documents_v2"

# Create a fresh persisted index from `chunks`
lc_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=lc_embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name,
)

lc_retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 6})


### Quick retrieval test

In [ ]:
lc_docs = lc_retriever.invoke("Abstract Factory?")
print(len(lc_docs), "documents")
print(lc_docs[0].page_content[:300] if lc_docs else "no hits")


In [ ]:
lc_docs

### Re-open the same DB later (without re-indexing)

If you want to reuse this exact DB later, do **not** call `from_documents` again. Instead:

In [ ]:
# lc_vectorstore = Chroma(
#     persist_directory=persist_directory,
#     collection_name=collection_name,
#     embedding_function=lc_embeddings,
# )
# lc_retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 6})
